# Generating synthetic news variants using an LLM

In [1]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

In [2]:
import pandas as pd

from dotenv import load_dotenv
from google import genai

In [3]:
from src.generation import (generate_synthetic_dataset)

In [4]:
PROCESSED_DIR = Path("data/processed")
TRAIN_PATH = PROCESSED_DIR / "train.csv"
SYNTHETIC_PATH = PROCESSED_DIR / "synthetic_train.csv"
AUGMENTED_PATH = PROCESSED_DIR / "augmented_train.csv"
TEST_PATH = PROCESSED_DIR / "test.csv"
CHALLENGE_PATH = PROCESSED_DIR / "challenge_test.csv"

In [5]:
train_df = pd.read_csv(TRAIN_PATH)

print("Number of training examples:",len(train_df))
train_df.head()

Number of training examples: 31268


,title,text,full_text,label,original_type,source_id
0,Philippine troops kill 14 Maoist rebels in cla...,MANILA ( ) - Philippine soldiers have killed 1...,Philippine troops kill 14 Maoist rebels in cla...,0,real,isot_13416
1,EU border controls could be extended in crisis...,BRUSSELS ( ) - Temporary border controls insid...,EU border controls could be extended in crisis...,0,real,isot_18702
2,Trump Whines About Hillary In Response To Obst...,After the news broke that Trump is going to be...,Trump Whines About Hillary In Response To Obst...,1,fake,isot_22321
3,Trump Throws PETTY Hissy Fit Because Bill Clin...,"Once again, Donald Trump is totally predictabl...",Trump Throws PETTY Hissy Fit Because Bill Clin...,1,fake,isot_24529
4,John Kerry commits more U.S. military aid for ...,TBILISI ( ) - Secretary of State John Kerry to...,John Kerry commits more U.S. military aid for ...,0,real,isot_08809


In [6]:
real_train = (
    train_df[train_df["label"] == 0]
    .copy()
    .reset_index(drop=True)
)
print("Number of real news:", len(real_train))
real_train.head()

Number of real news: 16955


,title,text,full_text,label,original_type,source_id
0,Philippine troops kill 14 Maoist rebels in cla...,MANILA ( ) - Philippine soldiers have killed 1...,Philippine troops kill 14 Maoist rebels in cla...,0,real,isot_13416
1,EU border controls could be extended in crisis...,BRUSSELS ( ) - Temporary border controls insid...,EU border controls could be extended in crisis...,0,real,isot_18702
2,John Kerry commits more U.S. military aid for ...,TBILISI ( ) - Secretary of State John Kerry to...,John Kerry commits more U.S. military aid for ...,0,real,isot_08809
3,Germany lauds anti-nuclear campaign winning No...,BERLIN ( ) - Germany on Friday welcomed the No...,Germany lauds anti-nuclear campaign winning No...,0,real,isot_17983
4,"In Cuba visit, Colorado governor sees governme...",HAVANA ( ) - The governor of Colorado said on ...,"In Cuba visit, Colorado governor sees governme...",0,real,isot_05598


In [7]:
# loading the Gemini API key

load_dotenv(".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY nije pronaden")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

GENERATION_MODEL = "gemini-3.5-flash-lite"

In [8]:
N_GENERATION_SOURCES = 1000

# selecting real training examples to use for synthetic news generation
generation_sources = (
    real_train
    .sample(
        n=min(N_GENERATION_SOURCES, len(real_train)),
        random_state=42
    )
    .reset_index(drop=True)
)

print("Number of original sources:", len(generation_sources))
print("Maximum number of synthetic examples:", len(generation_sources) * 3)

Number of original sources: 1000
Maximum number of synthetic examples: 3000


In [ ]:
# this cell was run to generate the synthetic training data and does not need to be run again

synthetic_train = generate_synthetic_dataset(
    source_df=generation_sources,
    client=client,
    model_name=GENERATION_MODEL
)

print("Total number of synthetic examples:", len(synthetic_train))

In [ ]:
# loading the existing synthetic_train.csv file
# (the generation cell above does not need to be run again)
synthetic_train = pd.read_csv(SYNTHETIC_PATH)

print("Number of loaded synthetic examples:", len(synthetic_train))

synthetic_train.head()

Number of loaded synthetic examples: 2997


,source_id,title,text,full_text,label,original_type,manipulation_type,change_summary,synthetic
0,isot_16792,Soccer star Weah to face vice president in Lib...,MONROVIA ( ) - Former soccer star George Weah ...,Soccer star Weah to face vice president in Lib...,1,synthetic,fact_change,Changed the vote margin lead from 10 points to...,True
1,isot_16792,You Won't Believe What This Soccer Legend Just...,MONROVIA ( ) - Former soccer star George Weah ...,You Won't Believe What This Soccer Legend Just...,1,synthetic,clickbait,"Rewrote the title into a dramatic, attention-g...",True
2,isot_16792,Shocking Showdown: Soccer Icon Weah Collides W...,MONROVIA ( ) - In a stunning political earthqu...,Shocking Showdown: Soccer Icon Weah Collides W...,1,synthetic,tone_shift,Shifted the tone of the entire article to be d...,True
3,isot_13888,Russia sends research ship to help search for ...,MOSCOW ( ) - Russia s defense ministry has sen...,Russia sends research ship to help search for ...,1,synthetic,fact_change,Changed the depth capability of the Yantar fro...,True
4,isot_13888,You Won't Believe Why Russia Just Sent This Ma...,MOSCOW ( ) - Russia s defense ministry has sen...,You Won't Believe Why Russia Just Sent This Ma...,1,synthetic,clickbait,Rewrote the title into a dramatic clickbait st...,True


In [10]:
synthetic_train = (
    synthetic_train
    .drop_duplicates(subset=["full_text"])
    .reset_index(drop=True)
)

In [ ]:
# run only after generating new synthetic data
synthetic_train.to_csv(
    SYNTHETIC_PATH,
    index=False
)

In [ ]:
# preparing the original training data
original_train = train_df.copy()

original_train["manipulation_type"] = "original"
original_train["change_summary"] = ""
original_train["synthetic"] = False
original_train["data_source"] = "original_train"

synthetic_train["data_source"] = "synthetic"

In [12]:
# defining the same columns for original and synthetic data

columns = [
    "source_id",
    "title",
    "text",
    "full_text",
    "label",
    "original_type",
    "manipulation_type",
    "change_summary",
    "synthetic",
    "data_source"
]

In [13]:
original_train = original_train[columns]
synthetic_train = synthetic_train[columns]

In [14]:
# combining original and synthetic training data
augmented_train = pd.concat(
    [original_train, synthetic_train],
    ignore_index=True
)

print("Original training examples:", len(original_train))
print("Synthetic examples:", len(synthetic_train))
print("Augmented training examples:", len(augmented_train))

Original training examples: 31268
Synthetic examples: 2997
Augmented training examples: 34265


In [15]:
# saving the augmented training set (does not need to be run again)

augmented_train.to_csv(
    AUGMENTED_PATH,
    index=False
)

print("Augmented training data saved")

Augmented training data saved


In [16]:
# loading the test set
test_df = pd.read_csv(TEST_PATH)

print("Number of test examples:", len(test_df))

Number of test examples: 7818


In [ ]:
# selecting only real news from the test set for challenge test generation
# fake test articles are excluded, the challenge test will contain real articles and LLM-generated fake variants

real_test = (
    test_df[test_df["label"] == 0]
    .copy()
    .reset_index(drop=True)
)

print("Number of real test articles:", len(real_test))

Number of real test articles: 4239


In [18]:
# selecting real test articles for the challenge test

SEED = 42
N_CHALLENGE_SOURCES = 200

challenge_sources = (
    real_test
    .sample(
        n=min(N_CHALLENGE_SOURCES, len(real_test)),
        random_state=SEED
    )
    .reset_index(drop=True)
)

print("Number of challenge sources:", len(challenge_sources))

Number of challenge sources: 200


In [19]:
# a different Gemini model is used for the challenge test to avoid learning the specific generator style seen during training

CHALLENGE_MODEL = "gemini-3.5-flash"

In [ ]:
# run once to generate the synthetic challenge test data (does not need to be run again)

challenge_synthetic = generate_synthetic_dataset(
    source_df=challenge_sources,
    client=client,
    model_name=CHALLENGE_MODEL
)

print("Number of generated challenge examples:", len(challenge_synthetic))

In [ ]:
# run only after generating new challenge data
challenge_synthetic["generator_model"] = CHALLENGE_MODEL
challenge_synthetic["synthetic"] = True

In [ ]:
# run only when creating a new challenge test
challenge_real = challenge_sources.copy()

challenge_real["manipulation_type"] = "original"
challenge_real["change_summary"] = ""
challenge_real["synthetic"] = False
challenge_real["generator_model"] = "none"

In [ ]:
# run only when creating a new challenge test
challenge_columns = [
    "source_id",
    "title",
    "text",
    "full_text",
    "label",
    "original_type",
    "manipulation_type",
    "change_summary",
    "synthetic",
    "generator_model"
]

In [ ]:
# run only when creating a new challenge test
challenge_real = challenge_real[challenge_columns]
challenge_synthetic = challenge_synthetic[challenge_columns]

In [ ]:
# run only when creating a new challenge test
challenge_test = pd.concat(
    [
        challenge_real,
        challenge_synthetic
    ],
    ignore_index=True
)

In [ ]:
# run only when creating a new challenge test
print("Original real:", len(challenge_real))
print("Synthetic manipulated:", len(challenge_synthetic))
print("Total challenge examples:", len(challenge_test))

In [ ]:
# run only when creating a new challenge test
challenge_test.to_csv(CHALLENGE_PATH, index=False)

print("Challenge test saved:", CHALLENGE_PATH)

In [ ]:
# run only when creating a new augmented training set
train_df = pd.read_csv("data/processed/train.csv")
synthetic_train = pd.read_csv("data/processed/synthetic_train.csv")

augmented_train = pd.concat(
    [train_df, synthetic_train],
    ignore_index=True
)

augmented_train.to_csv(
    "data/processed/augmented_train.csv",
    index=False
)

print("Training examples:", len(train_df))
print("Synthetic examples:", len(synthetic_train))
print("Augmented training examples:", len(augmented_train))

In [24]:
# loading the generated challenge test
challenge_test = pd.read_csv(CHALLENGE_PATH)

print("Number of saved challenge examples:", len(challenge_test))
print(challenge_test["manipulation_type"].value_counts())

challenge_test.head()

Number of saved challenge examples: 800
manipulation_type
original       200
fact_change    200
clickbait      200
tone_shift     200
Name: count, dtype: int64


,source_id,title,text,full_text,label,original_type,manipulation_type,change_summary,synthetic,generator_model
0,isot_15152,Cuban opposition falls at first hurdle as Cast...,HAVANA ( ) - Cuban opposition leaders said the...,Cuban opposition falls at first hurdle as Cast...,0,real,original,NaN,False,none
1,isot_06042,Ryan says Trump to address joint session of Co...,WASHINGTON ( ) - U.S. House of Representatives...,Ryan says Trump to address joint session of Co...,0,real,original,NaN,False,none
2,isot_04940,Senators want food safety review when U.S. fir...,WASHINGTON ( ) - Two U.S. senators from Midwes...,Senators want food safety review when U.S. fir...,0,real,original,NaN,False,none
3,isot_10625,House Republicans ready legal fight against Ob...,WASHINGTON ( ) - Republicans in the House of R...,House Republicans ready legal fight against Ob...,0,real,original,NaN,False,none
4,isot_04809,New York governor unveils South Bronx highway ...,( ) - New York Democratic Governor Andrew Cuom...,New York governor unveils South Bronx highway ...,0,real,original,NaN,False,none
